# Quoridor MCTS + LLM Optimization

**Quoridor** is a two-player adversarial board game well-suited to MCTS
(large branching factor due to wall placements, strategic depth).

## Design rationale (from the hand-tuned C++ reference in `quoridor/`)

| MCTS Phase | Hand-tuned? | Key Heuristic |
|---|---|---|
| **Selection** | No — standard UCT (C = √2) | — |
| **Expansion** | **Yes** | *Probable wall pruning* — only expand walls near existing walls/pawns (~15-30 instead of ~100+). When opponent has no walls left, only expand path-disturbing walls. |
| **Simulation** | **Yes** | *Biased rollout* — 70% shortest-path moves, 20% probable-wall placements, 10% exploration moves. BFS caching. 200-move cap with distance-based tiebreaker. |
| **Backprop** | No — standard alternating-player | — |

The LLM optimizer targets **expansion** and **simulation** — the two phases
where domain-specific heuristics make the biggest impact.

Configuration files:
- `mcts/hyperparams/quoridor_hyperparams.py` — engine params & optimization settings
- `mcts/training_logic/quoridor_training.py` — self-play levels & mastery criteria
- `LLM/game_infos/quoridor.txt` — game rules & strategy hints for the LLM

In [1]:
import sys
from pathlib import Path

ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from orchestrator import OptimizationRunner, Evaluator
from mcts import MCTSEngine
from mcts.games import Quoridor
from mcts.games.quoridor import PawnMove, HWall, VWall

print(f"Working dir: {Path('.').resolve()}")
print(f"Tool_Creation root: {ROOT}")
print("All imports OK ✓")

Working dir: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/scripts
Tool_Creation root: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation
All imports OK ✓


## 1. Game sanity check

Verify the Quoridor game state, legal actions, BFS distance, and wall placement.

In [2]:
game = Quoridor()
state = game.new_initial_state()
print(state)
print()

# Legal actions at start
actions = state.legal_actions()
pawn_moves = [a for a in actions if isinstance(a, PawnMove)]
h_walls = [a for a in actions if isinstance(a, HWall)]
v_walls = [a for a in actions if isinstance(a, VWall)]
print(f"Legal actions: {len(actions)} total = {len(pawn_moves)} pawn + "
      f"{len(h_walls)} h-walls + {len(v_walls)} v-walls")
print(f"Pawn moves: {pawn_moves}")
print()

# Play a few moves and verify BFS
state.apply_action(PawnMove(1, 4))   # P0 moves down
state.apply_action(PawnMove(7, 4))   # P1 moves up
state.apply_action(HWall(6, 3))      # P0 places a blocking wall
print("After 3 moves:")
print(state)
print(f"BFS: P0={state.shortest_dist(0)}, P1={state.shortest_dist(1)}")

# Wall correctly blocks?  P1 should have a longer path if wall is effective
assert state.shortest_dist(1) >= 8, "Wall should increase P1's path length"
print("Wall blocking verified ✓")

Turn 0 | Player 0
P0@(0, 4) walls=10  P1@(8, 4) walls=10
. . . . 0 . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . 1 . . . .

Legal actions: 3 total = 3 pawn + 0 h-walls + 0 v-walls
Pawn moves: [P(1,4), P(0,3), P(0,5)]

After 3 moves:
Turn 3 | Player 1
P0@(1, 4) walls=9  P1@(7, 4) walls=10
. . . . . . . . .
. . . . 0 . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . 1 . . . .
. . . . . . . . .
H-walls: [(6, 3)]
BFS: P0=8, P1=8
Wall blocking verified ✓


## 2. Baseline MCTS self-play

Run a single game with default MCTS tools and the Quoridor hyperparameters.
This measures how the untuned MCTS performs.

In [3]:
import importlib.util

# Load Quoridor hyperparams
_hp_path = ROOT / "mcts" / "hyperparams" / "quoridor_hyperparams.py"
_spec = importlib.util.spec_from_file_location("qhp", str(_hp_path))
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
hp = _mod.get_hyperparams()

ITERS = hp["iterations"]
MAX_DEPTH = hp["max_rollout_depth"]
PHASES = _mod.PHASES

print(f"Hyperparams: iterations={ITERS}, max_depth={MAX_DEPTH}, "
      f"C={hp.get('exploration_weight', 1.41)}")
print(f"Optimize phases: {PHASES}")

# Single baseline game
# NOTE: the default simulation in MCTS_tools/ is Sokoban-tuned, so
# we load the game-agnostic generic_simulation for Quoridor.
game = Quoridor()
engine = MCTSEngine(game, iterations=ITERS, max_rollout_depth=MAX_DEPTH, logging=True)
engine.load_tool("simulation", str(ROOT / "MCTS_tools" / "simulation" / "generic_simulation.py"))
baseline = engine.play_game(verbose=False)

tag = "P0 WIN" if baseline["solved"] else ("DRAW" if baseline["returns"] == [0.0, 0.0] else "P1 WIN")
print(f"\nBaseline: {tag} in {baseline['steps']} moves  returns={baseline['returns']}")
print(f"Trace: {baseline.get('log_file', 'N/A')}")

Hyperparams: iterations=200, max_depth=200, C=1.41
Optimize phases: ['simulation', 'expansion']

Baseline: P1 WIN in 38 moves  returns=[-1.0, 1.0]
Trace: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/mcts/records/quoridor_20260311_102448_482209.json


## 3. Batch baseline evaluation

Run multiple self-play games to establish a reliable baseline win rate.

In [4]:
import time

NUM_GAMES = 3
# Use fewer iterations for batch baseline (500 is slow — each game takes ~5 min)
BASELINE_ITERS = 200

game = Quoridor()
engine = MCTSEngine(game, iterations=BASELINE_ITERS, max_rollout_depth=MAX_DEPTH, logging=False)
engine.load_tool("simulation", str(ROOT / "MCTS_tools" / "simulation" / "generic_simulation.py"))

t0 = time.time()
stats = engine.play_many(num_games=NUM_GAMES, verbose=True)
elapsed = time.time() - t0

p0_wins = sum(1 for r in stats["results"] if r["returns"][0] > 0)
p1_wins = sum(1 for r in stats["results"] if r["returns"][1] > 0)
draws = NUM_GAMES - p0_wins - p1_wins

print(f"\nBaseline ({NUM_GAMES} games, {elapsed:.1f}s):")
print(f"  P0 wins: {p0_wins}  P1 wins: {p1_wins}  Draws: {draws}")
print(f"  Avg steps: {stats['avg_steps']}")
print(f"  P0 win rate: {p0_wins/NUM_GAMES:.0%}")

Move 1: action=P(0,3)
Turn 1 | Player 1
P0@(0, 3) walls=10  P1@(8, 4) walls=10
. . . 0 . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . 1 . . . .

Move 2: action=P(7,4)
Turn 2 | Player 0
P0@(0, 3) walls=10  P1@(7, 4) walls=10
. . . 0 . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . 1 . . . .
. . . . . . . . .

Move 3: action=P(0,4)
Turn 3 | Player 1
P0@(0, 4) walls=10  P1@(7, 4) walls=10
. . . . 0 . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . 1 . . . .
. . . . . . . . .

Move 4: action=V(2,2)
Turn 4 | Player 0
P0@(0, 4) walls=10  P1@(7, 4) walls=9
. . . . 0 . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . . . . . .
. . . . 1 . . . .
. . . . . . . . .
V-walls: [(2, 2)]

Move 5: action=

## 4. LLM optimization loop

Run the iterative LLM optimization targeting **expansion** and **simulation**.

The LLM receives:
- `LLM/game_infos/quoridor.txt` — game rules & heuristic hints (probable walls, BFS shortest path, etc.)
- Game traces from self-play
- Current expansion/simulation source code

It generates improved expansion and simulation functions that the evaluator tests against the baseline.

In [5]:
runner = OptimizationRunner.from_config(
    hyperparams_file="quoridor_hyperparams.py",
    verbose=True,
)

# The optimization loop can be very slow (each game ~2 min with 200 MCTS iters).
# Wrap in try/except so partial results are preserved if interrupted.
try:
    summary = runner.run()
except KeyboardInterrupt:
    print("\n⚠ Optimization interrupted — extracting partial results.")
    summary = {
        "all_results": runner.all_results,
        "best_fns": dict(runner.best_fns),
        "current_fns": dict(runner.current_fns),
        "current_hyperparams": dict(runner.current_hyperparams),
    }

best_fns = summary["best_fns"]
print(f"\nbest_fns: { {p: ('set' if f else 'None') for p, f in best_fns.items()} }")
print(f"Final hyperparams: {summary.get('current_hyperparams', {})}")

  Resuming simulation from previously installed tool.
Starting level: self_play
Hyperparams: iterations=200, max_depth=200, C=1.410
Phases to optimize: ['simulation', 'expansion']
  Computing baseline for self_play…

⚠ Optimization interrupted — extracting partial results.

best_fns: {'simulation': 'set', 'expansion': 'None'}
Final hyperparams: {'iterations': 200, 'max_rollout_depth': 200, 'exploration_weight': 1.41}


## 5. Comparative evaluation

Compare baseline (default tools) vs. optimized tools on self-play games.

In [ ]:
best_fns = summary["best_fns"]
opt_tools = {p: f for p, f in best_fns.items() if f is not None} or None

has_optimized = opt_tools is not None
print(f"Best fns: { {p: ('set' if f else 'None') for p, f in best_fns.items()} }")
print(f"Final hyperparams: {summary.get('current_hyperparams', {})}")

EVAL_N = 3

if not has_optimized:
    print("\nNo optimized functions adopted — skipping comparative eval.")
else:
    ev = runner.evaluator
    print(f"\nEvaluating self_play (n={EVAL_N} each) …")
    avg_b, sr_b, steps_b, _, t_b = ev.multi_eval(
        None, "self_play", n=EVAL_N, logging=False,
    )
    avg_o, sr_o, steps_o, _, t_o = ev.multi_eval(
        None, "self_play", n=EVAL_N, logging=False, extra_tools=opt_tools,
    )
    print(f"  Baseline  — P0 win rate: {sr_b:.0%}  avg_ret: {avg_b:.3f}  "
          f"avg_steps: {steps_b:.0f}  ({t_b:.1f}s)")
    print(f"  Optimized — P0 win rate: {sr_o:.0%}  avg_ret: {avg_o:.3f}  "
          f"avg_steps: {steps_o:.0f}  ({t_o:.1f}s)")

    delta = avg_o - avg_b
    print(f"\n  Δ avg_returns = {delta:+.3f}")

## 6. Inspect optimized tool source code

View the LLM-generated functions to see which Quoridor-specific
heuristics it discovered (e.g. probable wall pruning, BFS-guided rollouts).

In [ ]:
import inspect

for phase, fn in best_fns.items():
    print(f"{'='*60}")
    print(f"Phase: {phase}")
    print(f"{'='*60}")
    if fn is None:
        print("  (default — no LLM-generated function)")
    else:
        try:
            src = inspect.getsource(fn)
            print(src[:3000])
            if len(src) > 3000:
                print(f"\n... [{len(src) - 3000} more characters] ...")
        except (TypeError, OSError):
            print(f"  <callable: {fn}>")
    print()